<a href="https://colab.research.google.com/github/ddiaann/security-surveillance-using-drone/blob/main/FYP_fine_tuned_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Install required libraries

In [ ]:
!pip install open_clip_torch scikit-learn tqdm

2. import libraries

In [ ]:
import os, zipfile, random, shutil
import torch
import torch.nn.functional as F  # <--- THIS IS REQUIRED
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from PIL import Image
import open_clip
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
from tqdm import tqdm

3. upload dataset .zip

In [ ]:
from google.colab import files

uploaded = files.upload()

4. Extract dataset

In [ ]:
with zipfile.ZipFile("target_dataset.zip","r") as zip_ref:
    zip_ref.extractall("target_dataset")

print("Dataset extracted.")

5. Data augmentation

In [ ]:
input_folder = "target_dataset/target_dataset"
output_folder = "augmented_dataset"

os.makedirs(output_folder, exist_ok=True)

augmentation = transforms.Compose([

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomVerticalFlip(p=0.3),

    transforms.RandomApply([
        transforms.RandomRotation(30)
    ], p=0.7),

    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.4, contrast=0.4)
    ], p=0.5),

    transforms.RandomApply([
        transforms.ColorJitter(hue=0.1, saturation=0.4)
    ], p=0.5)

])

count = 0

for img_name in os.listdir(input_folder):

    if not img_name.lower().endswith((".jpg",".png",".jpeg")):
        continue

    img_path = os.path.join(input_folder, img_name)
    image = Image.open(img_path).convert("RGB")

    image.save(os.path.join(output_folder, f"img_{count}.jpg"))
    count += 1

    for i in range(8):

        aug_img = augmentation(image)

        aug_img.save(os.path.join(output_folder, f"img_{count}.jpg"))

        count += 1

print("Total augmented images:", count)

6. Split dataset 70:20:10

In [ ]:
dataset_root = "dataset"

train_dir = os.path.join(dataset_root,"train","target")
val_dir = os.path.join(dataset_root,"val","target")
test_dir = os.path.join(dataset_root,"test","target")

os.makedirs(train_dir,exist_ok=True)
os.makedirs(val_dir,exist_ok=True)
os.makedirs(test_dir,exist_ok=True)

images = os.listdir("augmented_dataset")
random.shuffle(images)

total = len(images)

train_split = int(0.7*total)
val_split = int(0.9*total)

train_imgs = images[:train_split]
val_imgs = images[train_split:val_split]
test_imgs = images[val_split:]

for img in train_imgs:
    shutil.copy(os.path.join("augmented_dataset",img),train_dir)

for img in val_imgs:
    shutil.copy(os.path.join("augmented_dataset",img),val_dir)

for img in test_imgs:
    shutil.copy(os.path.join("augmented_dataset",img),test_dir)

print("Train:",len(train_imgs))
print("Val:",len(val_imgs))
print("Test:",len(test_imgs))

7. load openCLIP model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)

tokenizer = open_clip.get_tokenizer("ViT-B-32")

model = model.to(device)

print("Model loaded")

8. create dataset objects

In [ ]:
train_dataset = datasets.ImageFolder("dataset/train", transform=preprocess_train)
val_dataset = datasets.ImageFolder("dataset/val", transform=preprocess_val)
test_dataset = datasets.ImageFolder("dataset/test", transform=preprocess_val)

print("Classes:", train_dataset.classes)

9. create data loaders

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=16,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=16,shuffle=False)
test_loader = DataLoader(test_dataset,batch_size=16,shuffle=False)

10. Define optimizer and loss

In [ ]:
import torch.nn as nn
import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=1e-5)

criterion = nn.CrossEntropyLoss()

template embedding

In [ ]:
# Make sure model is on device
model.eval()
template_embeddings = []

with torch.no_grad():
    for images, _ in train_loader:
        images = images.to(device)
        emb = model.encode_image(images)
        emb = F.normalize(emb, dim=-1)
        template_embeddings.append(emb)

# Average all embeddings to get single template
template = torch.mean(torch.cat(template_embeddings, dim=0), dim=0, keepdim=True)
template = F.normalize(template, dim=-1)

print("Template embedding created with shape:", template.shape)

11. Training loop (epochs and batches)

In [ ]:
# Multi-Epoch Training (Efficient Version)

epoch_list = [5, 10, 20, 50]
max_epochs = max(epoch_list)

results = pd.DataFrame(columns=["Epochs","Avg_Loss","Accuracy","Precision","Recall","F1-score"])

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

for epoch in range(max_epochs):

    model.train()
    running_loss = 0.0

    for images, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs}", leave=False):
        images = images.to(device)

        optimizer.zero_grad()

        image_features = model.encode_image(images)
        image_features = F.normalize(image_features, dim=-1)

        cosine_sim = torch.sum(image_features * template, dim=-1)

        loss = 1 - cosine_sim.mean()

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{max_epochs} Avg Loss: {avg_loss:.4f}")

    # Save checkpoint every epoch
    torch.save({
        'epoch': epoch+1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_loss
    }, f"checkpoint_epoch_{epoch+1}.pth")

    # Run evaluation only at selected epochs
    if (epoch+1) in epoch_list:

        model.eval()
        y_true = []
        y_pred = []
        threshold = 0.7

        with torch.no_grad():
            for images, _ in test_loader:
                images = images.to(device)

                image_features = model.encode_image(images)
                image_features = F.normalize(image_features, dim=-1)

                cosine_sim = torch.sum(image_features * template, dim=-1)

                preds = (cosine_sim >= threshold).int().cpu().numpy()

                y_pred.extend(preds)
                y_true.extend([1]*len(preds))

        acc = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)

        results = pd.concat([
            results,
            pd.DataFrame([{
                "Epochs": epoch+1,
                "Avg_Loss": avg_loss,
                "Accuracy": acc,
                "Precision": precision,
                "Recall": recall,
                "F1-score": f1
            }])
        ], ignore_index=True)

        # Save results every time
        results.to_csv("training_results.csv", index=False)

print(results)

In [ ]:
# 11️⃣ Multi-Epoch Training & Results Table
epoch_list = [5, 10, 20, 50]
results = pd.DataFrame(columns=["Epochs","Avg_Loss","Accuracy","Precision","Recall","F1-score"])

for num_epochs in epoch_list:
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for images, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
            images = images.to(device)
            optimizer.zero_grad()
            image_features = model.encode_image(images)
            image_features = F.normalize(image_features, dim=-1)
            cosine_sim = torch.sum(image_features * template, dim=-1)
            loss = 1 - cosine_sim.mean()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        avg_loss = running_loss / len(train_loader)
        print(f"[{num_epochs} Epochs] Epoch {epoch+1}/{num_epochs} Avg Loss: {avg_loss:.4f}")

    # Evaluate test set
    model.eval()
    y_true = []
    y_pred = []
    threshold = 0.7
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            image_features = model.encode_image(images)
            image_features = F.normalize(image_features, dim=-1)
            cosine_sim = torch.sum(image_features * template, dim=-1)
            preds = (cosine_sim >= threshold).int().cpu().numpy()
            y_pred.extend(preds)
            y_true.extend([1]*len(preds))

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    # Store results using pd.concat
    results = pd.concat([
        results,
        pd.DataFrame([{
            "Epochs": num_epochs,
            "Avg_Loss": avg_loss,
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1
        }])
    ], ignore_index=True)

print(results)

In [ ]:
model.eval()

y_true = []
y_pred = []
threshold = 0.7

with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)

        image_features = model.encode_image(images)
        image_features = F.normalize(image_features, dim=-1)

        cosine_sim = torch.sum(image_features * template, dim=-1)

        preds = (cosine_sim >= threshold).int().cpu().numpy()

        y_pred.extend(preds)
        y_true.extend([1]*len(preds))

acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

12. set model to evaluation mode

In [ ]:
model.eval()

13. Evaluate on test set

In [ ]:
y_true = []
y_pred = []

threshold = 0.7  # similarity threshold

with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        image_features = model.encode_image(images)
        image_features = F.normalize(image_features, dim=-1)
        cosine_sim = torch.sum(image_features * template, dim=-1)

        preds = (cosine_sim >= threshold).int().cpu().numpy()
        y_pred.extend(preds)
        y_true.extend([1]*len(preds))  # all test images are the same person

14. Compute performance metrics

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("Test Accuracy:", round(accuracy, 4))
print("Test Precision:", round(precision, 4))
print("Test Recall:", round(recall, 4))
print("Test F1-score:", round(f1, 4))
print("\nConfusion Matrix:\n", cm)

15. Save the trained model

In [ ]:
torch.save(model.state_dict(), "finetuned_single_person_openclip.pt")
print("Model saved as 'finetuned_single_person_openclip.pt'")

16. Download model to laptop

In [ ]:
from google.colab import files
files.download("finetuned_single_person_openclip.pt")

17. Save results table

In [ ]:
results.to_csv("epoch_results.csv", index=False)
files.download("epoch_results.csv")